# **LLM Meal Planner**

In [1]:
import os
import sys
from datetime import datetime, timedelta

from openai import OpenAI

sys.path.insert(0, os.path.abspath(".."))

import json
from random import randint, random, sample, seed

from dotenv import load_dotenv

from engines import LLMMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import (
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    Recipe,
)
from models.DietaryTag import DietaryTag

load_dotenv()

True

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-05-01
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-04
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-06
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
with open("../recipe_extraction/supplemented_structured_recipes.json", "r") as file:
    all_recipes = json.load(file)

In [12]:
NUM_EXTRA_RECIPES = 10

In [13]:
filtered_recipes_indices = []

for ingredient_name in pantry_ingredient_name_by_quantity.keys():
    for i in range(len(all_recipes)):
        if ingredient_name in [ingredient["ingredient"] for ingredient in all_recipes[i]["ingredients"]]:
            filtered_recipes_indices.append(i)

# sample some more recipes that do not contain any ingredients from the pantry
sampled_recipes_indices = sample(
    [i for i in range(len(all_recipes)) if i not in filtered_recipes_indices],
    NUM_EXTRA_RECIPES,
)

In [14]:
assert len(set(filtered_recipes_indices).intersection(set(sampled_recipes_indices))) == 0

final_recipes = [all_recipes[i] for i in filtered_recipes_indices + sampled_recipes_indices]

In [15]:
tag_map = {
    "VEGETARIAN": DietaryTag.VEGETARIAN,
    "VEGAN": DietaryTag.VEGAN,
    "GLUTEN_FREE": DietaryTag.GLUTEN_FREE,
    "LACTOSE_FREE": DietaryTag.LACTOSE_FREE,
}

recipe_objects = []

for raw_recipe in final_recipes:
    ingredients = {ingredient["ingredient"]: ingredient["quantity"] for ingredient in raw_recipe["ingredients"]}
    dietary_tags = [tag_map[tag] for tag in raw_recipe.get("dietary_tags", []) if tag in tag_map]
    nutritional_information = raw_recipe["nutritional_information"]

    recipe = Recipe(
        name=raw_recipe["name"].strip(),
        ingredients=ingredients,
        dietary_tags=dietary_tags,
        instructions=raw_recipe.get("instructions", []),
    )

    recipe.nutritional_information = NutritionalInformation(
        calories=nutritional_information.get("calories"),
        carbohydrates=nutritional_information.get("carbohydrates"),
        sugar=nutritional_information.get("sugar"),
        protein=nutritional_information.get("protein"),
        fat=nutritional_information.get("fat"),
        saturated_fat=nutritional_information.get("saturated_fat"),
        fiber=nutritional_information.get("fiber"),
        sodium=nutritional_information.get("sodium"),
        is_gluten_free=nutritional_information.get("is_gluten_free"),
        is_lactose_free=nutritional_information.get("is_lactose_free"),
        is_vegetarian=nutritional_information.get("is_vegetarian"),
        is_vegan=nutritional_information.get("is_vegan"),
    )

    recipe_objects.append(recipe)

final_recipes = recipe_objects

In [16]:
print(f"Number of recipes before filtering: {len(all_recipes)}")
print(f"Number of recipes after filtering: {len(final_recipes)}")

Number of recipes before filtering: 19716
Number of recipes after filtering: 138


In [17]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=final_recipes,
    pantry=pantry,
    preferences=preferences,
)

In [18]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [19]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [20]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [21]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

planner = LLMMealPlanner(meal_planning_environment, llm_client=client, prompt_filepath="./LLMMealPlannerPrompt.txt")

In [22]:
print(planner.prompt)

You are a meal planning assistant. Your task is to select a 7-day meal plan (3 meals per day = 21 meals total) from a provided list of recipes, optimised according to the user's preferences and pantry.

You will be given:
1. A numbered list of available recipes (index, name, dietary tags, calories, protein, key ingredients).
2. The user's pantry: ingredients currently in stock with their available quantities (in grams), with days until expiry and estimated financial cost.
3. The user's preferences: daily calorie target, daily protein target, weekly grocery budget, and any dietary restrictions (vegetarian, vegan, gluten-free, lactose-free).

This list of input should always remain untouched. You must not add any new recipes or ingredients, and you must not modify the pantry or user preferences. Your task is solely to select the optimal combination of recipes from the provided list based on the given information.

---

YOUR GOAL

Select exactly 21 recipe indices (integers) from the provi

In [23]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan()

In [28]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,"Beef, Mushroom, and Broccoli Stir-Fry",Chinese Orange Chicken,"Beef, Mushroom, and Broccoli Stir-Fry"
1,2,Lamb Stew with Black Mustard Seeds,"Beef, Mushroom, and Broccoli Stir-Fry",Beef and Green Pepper Stir-Fry
2,3,Baked Lemon Shrimp with Garlic,Lime-Scented Rice,Steamed Rice
3,4,Chicken Adobo,Green-Curry Chicken with Peas and Basil,Green Poblano Rice (Arroz Verde al Poblano)
4,5,Casserole of Turkey with Rice,Yellow Rice,Steamed Rice
5,6,Asian-Style Shrimp and Pineapple Fried Rice,"Fried Rice with Ham, Egg, and Scallions",Lime-Scented Rice
6,7,The World of Rice Salads,Lime-Scented Rice,Steamed Rice


In [29]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.000000,800.000000,1d
1,broccoli,1500,680.388555,819.611445,4d
2,rice,1000,874.413060,125.586940,6d


In [30]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 99
Total cost: €31.58


,Ingredient,Quantity to Buy (g),Cost (€)
0,Butter,4.4,0.04
1,Chinese Shaoxing wine,1.6,0.01
2,Equipment,33.3,0.06
3,Mango Chutney,20.0,0.05
4,Salt,2.1,0.01
...,...,...,...
95,white onion,14.3,0.03
96,white wine,5.5,0.00
97,white wine vinegar,23.7,0.20
98,whole cloves,28.6,0.05


In [31]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,2004.4 kcal,124.7 g,4.4 kcal,74.7 g
Day 2,2220.0 kcal,119.5 g,220.0 kcal,69.5 g
Day 3,2024.6 kcal,249.8 g,24.6 kcal,199.8 g
Day 4,872.6 kcal,56.0 g,-1127.4 kcal,6.0 g
Day 5,840.7 kcal,49.4 g,-1159.3 kcal,-0.6 g
Day 6,999.6 kcal,33.6 g,-1000.4 kcal,-16.4 g
Day 7,599.8 kcal,10.6 g,-1400.2 kcal,-39.4 g
